# 🔍 query.ipynb — Ask Questions with RAG
### RAG Project | Step 2 of 3

This notebook handles the **Querying Phase**:
1. Load the FAISS index built by `ingest.ipynb`
2. Embed the user question using the same HuggingFace model
3. Retrieve the top-k most relevant chunks from FAISS
4. Send context + question to **Groq (Llama3)** for a grounded answer
5. Display the answer with source citations

> ⚠️ Run `ingest.ipynb` first to build the FAISS index before running this notebook.


## Step 1 — Imports & Groq API Key

In [ ]:
import os
import textwrap

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
os.environ["GROQ_API_KEY"] = "your-groq-api-key"


print(" Imports successful!")

d:\ai rag project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Imports successful!


##  Step 2 — Configuration

In [2]:
# ── CONFIGURE HERE ────────────────────────────────────────────────────────
INDEX_DIR  = "faiss_index"      # Must match INDEX_DIR from ingest.ipynb

GROQ_MODEL = "llama-3.3-70b-versatile"  # Options:
                                #   "llama3-8b-8192"     → fastest, good quality
                                #   "llama3-70b-8192"    → best quality, slightly slower
                                #   "mixtral-8x7b-32768" → largest context (32k tokens)
                                #   "gemma2-9b-it"       → Google Gemma, very fast

K_CHUNKS   = 3                  # Number of chunks to retrieve (increase for longer answers)

print(f" Index:  {INDEX_DIR}/")
print(f" Model:  {GROQ_MODEL}")
print(f" Top-k:  {K_CHUNKS} chunks")

 Index:  faiss_index/
 Model:  llama-3.3-70b-versatile
 Top-k:  3 chunks


##  Step 3 — Load Embedding Model & FAISS Index

In [3]:
# Must use the SAME embedding model as ingest.ipynb
print(" Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")

print(" Loading FAISS index...")
vectorstore = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True  # safe — we created this index ourselves
)
print(f" FAISS index loaded")
print(f"   Vectors in index: {vectorstore.index.ntotal}")

 Loading embedding model...


C:\Users\KRISHA\AppData\Local\Temp\ipykernel_22220\1065879328.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11791.72it/s]


Embedding model loaded
 Loading FAISS index...
 FAISS index loaded
   Vectors in index: 3


## Step 4 — Define the RAG Prompt

The prompt template is the most important part of RAG.  
It **forces the LLM to answer only from retrieved context** — this prevents hallucination.


In [4]:
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are a helpful assistant. Use ONLY the context below to answer.\n"
        "If the answer is not in the context, say: 'I don't know based on this document.'\n"
        "Do NOT make up information.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    )
)

print(" Prompt template defined")
print("\n--- Template Preview ---")
print(RAG_PROMPT.template)

 Prompt template defined

--- Template Preview ---
You are a helpful assistant. Use ONLY the context below to answer.
If the answer is not in the context, say: 'I don't know based on this document.'
Do NOT make up information.

Context:
{context}

Question: {question}

Answer:


##  Step 5 — Build the RAG Chain

In [5]:
# # Retriever: fetches top-k chunks from FAISS
# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": K_CHUNKS}
# )

# # LLM: Groq hosted Llama3 (ultra fast inference)
# llm = ChatGroq(
#     model=GROQ_MODEL,
#     temperature=0,   # 0 = deterministic answers, no creativity
# )

# # Full RAG chain: retriever + prompt + LLM
# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     retriever=retriever,
#     chain_type="stuff",                        # "stuff" = join all chunks into one prompt
#     chain_type_kwargs={"prompt": RAG_PROMPT},
#     return_source_documents=True               # enables source citations
# )

# print(f"✅ RAG chain ready with model: {GROQ_MODEL}")
# print("\n You can now ask questions in the cells below!")

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# ── LLM ───────────────────────────────────────────────────────────────────
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── Build chain using modern LCEL (LangChain Expression Language) ─────────
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def final_step(x):
    response = llm.invoke(
        RAG_PROMPT.format(
            context=format_docs(x["docs"]),
            question=x["question"]
        )
    )
    
    return {
        "answer": response.content,
        "source_documents": x["docs"]
    }

rag_chain = (
    {
        "docs": itemgetter("input") | retriever,
        "question": itemgetter("input")
    }
    | RunnableLambda(final_step)   # ✅ FIX HERE
)

##  Step 6 — Ask Your First Question

Change `QUESTION` to anything relevant to the document you indexed.


In [6]:
question = "What is the difference between supervised and unsupervised learning?"

# Run chain
result = rag_chain.invoke({"input": question})

print("=" * 65)
print(f" Question: {question}")
print("=" * 65)
result = rag_chain.invoke({"input": question})

print("\n Answer:\n")
print(result["answer"])

print("\n Sources:")
for i, doc in enumerate(result["source_documents"], 1):
    page = doc.metadata.get("page", "?")
    snippet = doc.page_content[:200].replace("\n", " ")
    
    print(f"\n[{i}] Page {page}:")
    print(f"   {snippet}...")


 Question: What is the difference between supervised and unsupervised learning?

 Answer:

Supervised Learning is trained on labelled data, whereas Unsupervised Learning finds patterns in unlabelled data.

 Sources:

[1] Page 0:
   Unsupervised Learning: Finds patterns in unlabelled data. Example: customer clustering. Reinforcement Learning: Learns via rewards and penalties. Example: AlphaGo. 3. Deep Learning Uses neural network...

[2] Page 0:
   Artificial Intelligence: A Beginner's Guide Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to think, learn, and solve problems like humans. 1. Types...

[3] Page 0:
   5. Real-World Applications Healthcare: AI diagnoses diseases from medical images. Finance: Fraud detection. Transportation: Self-driving cars. Retail: Recommendation engines. 6. Challenges Key challen...


##  Step 7 — Reusable ask() Function

In [7]:
def ask(question: str, show_sources: bool = True):

    answer = rag_chain.invoke({"input": question})   # ✅ FIX
    docs   = retriever.invoke(question)              # this is fine

    print("=" * 65)
    print(f"  {question}")
    print("-" * 65)
    print(f"  {answer['answer']}")   # since now it's dict

    if show_sources:
        print("\n  Sources:")
        for i, doc in enumerate(answer["source_documents"], 1):
            page = doc.metadata.get("page", "?")
            snippet = doc.page_content[:150].replace("\n", " ")
            print(f"   [{i}] Page {page}: {snippet}...")

    print()

# ── Test with multiple questions ──────────────────────────────────────────
ask("What is deep learning?")
ask("What are the real-world applications of AI?")
ask("What is reinforcement learning?")

  What is deep learning?
-----------------------------------------------------------------
  Deep Learning uses neural networks with many layers. It excels at image recognition, NLP, and autonomous driving.

  Sources:
   [1] Page 0: Unsupervised Learning: Finds patterns in unlabelled data. Example: customer clustering. Reinforcement Learning: Learns via rewards and penalties. Exam...
   [2] Page 0: Artificial Intelligence: A Beginner's Guide Artificial Intelligence (AI) refers to the simulation of human intelligence in machines programmed to thin...
   [3] Page 0: 5. Real-World Applications Healthcare: AI diagnoses diseases from medical images. Finance: Fraud detection. Transportation: Self-driving cars. Retail:...

  What are the real-world applications of AI?
-----------------------------------------------------------------
  The real-world applications of AI include: 
1. Healthcare: AI diagnoses diseases from medical images.
2. Finance: Fraud detection.
3. Transportation: Self-driv

##  Step 8 — Inspect Raw Similarity Search

See the raw chunks FAISS retrieves **before** they go to the LLM.  
Useful for debugging when answers seem wrong — check if the right chunks are being fetched.


In [8]:
query = "What are the applications of artificial intelligence?"

# Returns list of (Document, score) — lower score = more similar in FAISS L2
results = vectorstore.similarity_search_with_score(query, k=4)

print(f"Top {len(results)} chunks for:\n'{query}'\n")
print("=" * 65)

for i, (doc, score) in enumerate(results, 1):
    page = doc.metadata.get("page", "?")
    print(f"\n[Chunk {i}] Page {page} | L2 distance score: {score:.4f}")
    print("-" * 40)
    print(textwrap.fill(doc.page_content, width=80))

print("\n💡 Lower score = more similar to your query")

Top 3 chunks for:
'What are the applications of artificial intelligence?'


[Chunk 1] Page 0 | L2 distance score: 0.7485
----------------------------------------
Artificial Intelligence: A Beginner's Guide Artificial Intelligence (AI) refers
to the simulation of human intelligence in machines programmed to think, learn,
and solve problems like humans. 1. Types of AI Narrow AI: Designed for specific
tasks like facial recognition or search. Examples: Siri, Alexa. General AI:
Hypothetical AI that performs any intellectual task. Does not exist yet. 2.
Machine Learning Supervised Learning: Trained on labelled data. Example: email
spam detection.

[Chunk 2] Page 0 | L2 distance score: 1.0511
----------------------------------------
Unsupervised Learning: Finds patterns in unlabelled data. Example: customer
clustering. Reinforcement Learning: Learns via rewards and penalties. Example:
AlphaGo. 3. Deep Learning Uses neural networks with many layers. Excels at image
recognition, NLP, and autono